# Chạy thử nhanh: feature selection → Random Forest (1 fold)

Chọn `N_FEATURES` gene trên **train** của fold `FOLD` bằng thuật toán `METHOD`, train Random Forest trên các gene đó
rồi đánh giá trên **val** và **test**. So sánh: có/không SMOTE (chỉ áp trên train, sau bước chọn gene),
RF dùng toàn bộ gene và mốc "luôn đoán lớp đông nhất".
Dữ liệu giữ nguyên log2, không chuẩn hoá. Đổi `METHOD` ở cell cấu hình để thử thuật toán khác.

In [ ]:
import os, time, warnings
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix, accuracy_score,
                             balanced_accuracy_score, f1_score)

warnings.filterwarnings("ignore")

# ====== CAU HINH (sua tay) ======
DATA_DIR = "/kaggle/input/datasets/lfreedom2750/experiment-dataset/h5/gse68465"
FOLD = 1
N_FEATURES = 100
METHOD = "f_classif"      # "mutual_info" | "f_classif" | "mrmr" | "relieff" | "multisurf"
SEED = 42

# Tham so rieng tung thuat toan (giong notebook chinh)
MI_NEIGHBORS = 3
RELIEF_K = 10
MRMR_RELEVANCE, MRMR_REDUNDANCY, MRMR_DENOMINATOR = "f", "c", "mean"

RF_PARAMS = dict(n_estimators=500, max_features="sqrt", min_samples_leaf=1,
                 class_weight="balanced", n_jobs=-1, random_state=SEED)
SMOTE_K = 5               # k_neighbors cua SMOTE (tu giam neu lop qua it mau)
SMOTE_ALL_GENES = True    # them cau hinh "tat ca gene + SMOTE" de so sanh (cham hon)
STAGE_NAMES = {0: "Normal", 1: "I", 2: "II", 3: "III", 4: "IV"}

if METHOD == "mrmr":
    get_ipython().system("pip install -q mrmr_selection")
get_ipython().system("pip install -q tables imbalanced-learn")

## Đọc dữ liệu

In [ ]:
def load_split(split):
    df = pd.read_hdf(os.path.join(DATA_DIR, f"fold_{FOLD}", f"{split}.h5"), key="data")
    X = df.drop(columns="label")
    return X.values.astype(np.float64), df["label"].values.astype(int), X.columns.to_numpy()


(X_tr, y_tr, genes), (X_va, y_va, _), (X_te, y_te, _) = [load_split(s) for s in ["train", "val", "test"]]
for name, X, y in [("train", X_tr, y_tr), ("val", X_va, y_va), ("test", X_te, y_te)]:
    print(f"{name:5s} {X.shape} | {pd.Series(y).map(STAGE_NAMES).value_counts().to_dict()}")

## Các thuật toán chọn đặc trưng (giống notebook chính)

In [ ]:
def top_k(scores, k):
    scores = np.nan_to_num(np.asarray(scores, dtype=float), nan=-np.inf)
    idx = np.argsort(-scores, kind="stable")[:k]
    return idx, scores[idx]

from sklearn.feature_selection import mutual_info_classif


def sel_mutual_info(X, y, k):
    kw = dict(discrete_features=False, n_neighbors=MI_NEIGHBORS, random_state=SEED)
    try:
        mi = mutual_info_classif(X, y, n_jobs=-1, **kw)   # sklearn >= 1.5
    except TypeError:
        mi = mutual_info_classif(X, y, **kw)
    return top_k(mi, k)

from sklearn.feature_selection import f_classif


def sel_f_classif(X, y, k):
    F, _ = f_classif(X, y)      # gene hang so -> F = NaN, bi day xuong cuoi
    return top_k(F, k)

def sel_mrmr(X, y, k):
    from mrmr import mrmr_classif
    Xdf = pd.DataFrame(X, columns=np.arange(X.shape[1]))   # ten cot = chi so, map lai gene sau
    sel, relevance, _ = mrmr_classif(X=Xdf, y=pd.Series(y), K=k, relevance=MRMR_RELEVANCE,
                             redundancy=MRMR_REDUNDANCY, denominator=MRMR_DENOMINATOR,
                             return_scores=True, n_jobs=-1, show_progress=False)
    idx = np.asarray(sel, dtype=int)
    # rank = thu tu mRMR chon; score = relevance (F-statistic) cua gene do
    return idx, relevance.loc[idx].to_numpy(dtype=float)

def _scaled_diff_matrix(X):
    # diff(f, a, b) = |a - b| / range(f); gene hang so -> diff = 0
    rng = X.max(0) - X.min(0)
    rng[rng == 0] = np.inf
    Xs = X / rng
    n = len(Xs)
    D = np.empty((n, n))
    for i in range(n):
        D[i] = np.abs(Xs - Xs[i]).sum(1)
    return Xs, D


def relieff_scores(X, y, k=10):
    Xs, D = _scaled_diff_matrix(X)
    n, p = Xs.shape
    classes, counts = np.unique(y, return_counts=True)
    prior = dict(zip(classes, counts / n))
    W = np.zeros(p)
    for i in range(n):
        d = D[i].copy()
        d[i] = np.inf
        ci = y[i]
        hits = np.flatnonzero(y == ci)
        hits = hits[hits != i]
        if len(hits):
            hits = hits[np.argsort(d[hits], kind="stable")[:k]]
            W -= np.abs(Xs[hits] - Xs[i]).mean(0)
        for c in classes:
            if c == ci:
                continue
            miss = np.flatnonzero(y == c)
            miss = miss[np.argsort(d[miss], kind="stable")[:k]]
            W += prior[c] / (1 - prior[ci]) * np.abs(Xs[miss] - Xs[i]).mean(0)
    return W / n


def sel_relieff(X, y, k):
    return top_k(relieff_scores(X, y, RELIEF_K), k)

def _scaled_diff_matrix(X):
    # diff(f, a, b) = |a - b| / range(f); gene hang so -> diff = 0
    rng = X.max(0) - X.min(0)
    rng[rng == 0] = np.inf
    Xs = X / rng
    n = len(Xs)
    D = np.empty((n, n))
    for i in range(n):
        D[i] = np.abs(Xs - Xs[i]).sum(1)
    return Xs, D


def multisurf_scores(X, y):
    Xs, D = _scaled_diff_matrix(X)
    n, p = Xs.shape
    classes, counts = np.unique(y, return_counts=True)
    prior = dict(zip(classes, counts / n))
    W = np.zeros(p)
    for i in range(n):
        others = np.arange(n) != i
        d = D[i]
        T = d[others].mean() - d[others].std() / 2
        near = np.flatnonzero(others & (d < T))
        ci = y[i]
        hits = near[y[near] == ci]
        if len(hits):
            W -= np.abs(Xs[hits] - Xs[i]).mean(0)
        for c in classes:
            if c == ci:
                continue
            miss = near[y[near] == c]
            if len(miss):
                W += prior[c] / (1 - prior[ci]) * np.abs(Xs[miss] - Xs[i]).mean(0)
    return W / n


def sel_multisurf(X, y, k):
    return top_k(multisurf_scores(X, y), k)

## Chọn đặc trưng trên train

In [ ]:
SELECTORS = {name: fn for name, fn in [
    ("mutual_info", sel_mutual_info), ("f_classif", sel_f_classif), ("mrmr", sel_mrmr),
    ("relieff", sel_relieff), ("multisurf", sel_multisurf)]}

t0 = time.perf_counter()
idx, scores = SELECTORS[METHOD](X_tr, y_tr, N_FEATURES)   # chi dung tap train
fs_time = time.perf_counter() - t0
sel_genes = genes[idx]
print(f"{METHOD}: chon {len(idx)} gene trong {fs_time:.2f}s")
print("Top 20:", list(sel_genes[:20]))

## Random Forest: có/không SMOTE, gene đã chọn vs toàn bộ gene

In [ ]:
from imblearn.over_sampling import SMOTE


def apply_smote(X, y):
    # Chi ap tren TRAIN (sau khi chon gene); val/test giu nguyen mau that
    counts = pd.Series(y).value_counts()
    k = int(min(SMOTE_K, counts.min() - 1))   # lop it mau hon SMOTE_K+1 -> giam k_neighbors
    if k < 1:
        print(f"[SMOTE] lop nho nhat chi co {counts.min()} mau -> bo qua SMOTE")
        return X, y
    Xs, ys = SMOTE(k_neighbors=k, random_state=SEED).fit_resample(X, y)
    print(f"[SMOTE] k_neighbors={k} | {len(y)} -> {len(ys)} mau | "
          f"{pd.Series(ys).map(STAGE_NAMES).value_counts().to_dict()}")
    return Xs, ys


def fit_rf(X, y):
    return RandomForestClassifier(**RF_PARAMS).fit(X, y)


all_idx = np.arange(X_tr.shape[1])
X_sm, y_sm = apply_smote(X_tr[:, idx], y_tr)
MODELS = {
    f"{METHOD} top{N_FEATURES}": (idx, fit_rf(X_tr[:, idx], y_tr)),
    f"{METHOD} top{N_FEATURES} + SMOTE": (idx, fit_rf(X_sm, y_sm)),
    "tat ca gene": (all_idx, fit_rf(X_tr, y_tr)),
}
if SMOTE_ALL_GENES:
    MODELS["tat ca gene + SMOTE"] = (all_idx, fit_rf(*apply_smote(X_tr, y_tr)))

rows, PRED_TEST = [], {}
for split, X_eval, y_eval in [("val", X_va, y_va), ("test", X_te, y_te)]:
    for tag, (cols, rf) in MODELS.items():
        pred = rf.predict(X_eval[:, cols])
        if split == "test":
            PRED_TEST[tag] = pred
        rows.append({"split": split, "setting": tag, "n_genes": len(cols),
                     "accuracy": accuracy_score(y_eval, pred),
                     "balanced_acc": balanced_accuracy_score(y_eval, pred),
                     "macro_f1": f1_score(y_eval, pred, average="macro", zero_division=0),
                     "weighted_f1": f1_score(y_eval, pred, average="weighted", zero_division=0)})

# Moc so sanh: luon doan lop dong nhat cua train
majority = pd.Series(y_tr).mode()[0]
print(f"Accuracy khi luon doan lop dong nhat ({STAGE_NAMES[majority]}): test = {np.mean(y_te == majority):.3f}")
pd.DataFrame(rows).round(4)

## Chi tiết trên test

In [ ]:
for tag in [f"{METHOD} top{N_FEATURES}", f"{METHOD} top{N_FEATURES} + SMOTE"]:
    pred = PRED_TEST[tag]
    labels = sorted(set(y_te) | set(pred))
    names = [STAGE_NAMES[l] for l in labels]
    print(f"\n--- {tag} / fold {FOLD} / test ---")
    print(classification_report(y_te, pred, labels=labels, target_names=names, digits=3, zero_division=0))
    display(pd.DataFrame(confusion_matrix(y_te, pred, labels=labels),
                         index=[f"that_{n}" for n in names], columns=[f"du_doan_{n}" for n in names]))